In [1]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path

RAW_PATH = Path("../data/empleados_analitica.csv")
DB_PATH = Path("../data/empleados.db")
SCHEMA_PATH = Path("../sql/schema.sql")

df = pd.read_csv(RAW_PATH)

print("Dimensiones:", df.shape)
print("\nTipos de datos:\n", df.dtypes)
display(df.head())

print("\nNulos por columna:\n", df.isna().sum())
print("\nDuplicados exactos:", df.duplicated().sum())
print("Duplicados por id_empleado:", df["id_empleado"].duplicated().sum())
display(df.describe().T)

Dimensiones: (603, 11)

Tipos de datos:
 id_empleado               int64
edad                    float64
experiencia_anios         int64
nivel_educativo           int64
departamento                str
ciudad                      str
genero                      str
horas_extra_mes           int64
satisfaccion_laboral      int64
antiguedad_empresa        int64
salario_mensual         float64
dtype: object


,id_empleado,edad,experiencia_anios,nivel_educativo,departamento,ciudad,genero,horas_extra_mes,satisfaccion_laboral,antiguedad_empresa,salario_mensual
0,1,26.0,0,4,Ventas,CDMX,M,7,1,0,27253.955381
1,2,41.0,19,2,Operaciones,CDMX,F,11,5,18,41513.348881
2,3,42.0,20,2,Marketing,CDMX,M,21,3,18,45448.151836
3,4,52.0,30,4,Finanzas,CDMX,M,20,1,30,62117.356776
4,5,57.0,31,1,IT,CDMX,M,9,1,30,55385.803611



Nulos por columna:
 id_empleado              0
edad                     8
experiencia_anios        0
nivel_educativo          0
departamento             0
ciudad                   0
genero                   0
horas_extra_mes          0
satisfaccion_laboral     0
antiguedad_empresa       0
salario_mensual         15
dtype: int64

Duplicados exactos: 3
Duplicados por id_empleado: 3


,count,mean,std,min,25%,50%,75%,max
id_empleado,603.0,299.034826,174.156144,1.000000,148.500000,299.000000,449.500000,600.0
edad,595.0,41.151261,10.908279,22.000000,32.000000,42.000000,50.000000,60.0
experiencia_anios,603.0,16.734660,10.727177,0.000000,7.500000,17.000000,25.000000,35.0
nivel_educativo,603.0,2.479270,1.140096,1.000000,1.000000,2.000000,4.000000,4.0
horas_extra_mes,603.0,10.587065,6.400293,0.000000,5.000000,10.000000,16.000000,21.0
satisfaccion_laboral,603.0,3.077944,1.429016,1.000000,2.000000,3.000000,4.000000,5.0
antiguedad_empresa,603.0,15.127695,10.185421,0.000000,6.000000,16.000000,24.000000,30.0
salario_mensual,588.0,42931.656939,13395.066155,17779.985779,33337.427488,42387.220155,52121.667879,210000.0


In [2]:
df_clean = df.copy()

# 1. Eliminar duplicados conservando el primer registro
df_clean = df_clean.drop_duplicates(subset=["id_empleado"], keep="first").reset_index(drop=True)

# 2. Normalizar textos categóricos (por buena práctica y reproducibilidad)
for col in ["departamento", "ciudad", "genero"]:
    df_clean[col] = df_clean[col].astype(str).str.strip()

# 3. Imputación de valores faltantes
# Edad: mediana por departamento (redondeada a entero)
df_clean["edad"] = (
    df_clean.groupby("departamento")["edad"]
    .transform(lambda s: s.fillna(s.median()))
    .round()
    .astype(int)
)

# Salario: mediana por departamento (menos sensible al outlier de $210,000 que la media)
df_clean["salario_mensual"] = df_clean.groupby("departamento")["salario_mensual"].transform(
    lambda s: s.fillna(s.median())
)

# 4. Tratamiento de valores atípicos en salario_mensual (Criterio: IQR con tope / winsorizing)
Q1 = df_clean["salario_mensual"].quantile(0.25)
Q3 = df_clean["salario_mensual"].quantile(0.75)
IQR = Q3 - Q1
lim_inf = max(0, Q1 - 1.5 * IQR)
lim_sup = Q3 + 1.5 * IQR

outliers_antes = (df_clean["salario_mensual"] > lim_sup).sum()
df_clean["salario_mensual"] = df_clean["salario_mensual"].clip(lower=lim_inf, upper=lim_sup).round(2)

print(f"Límite superior IQR: {lim_sup:,.2f} | Outliers recortados: {outliers_antes}")
print("Nulos restantes:", df_clean.isna().sum().sum())

Límite superior IQR: 79,107.94 | Outliers recortados: 1
Nulos restantes: 0


In [3]:
# 1. Rango de edad
df_clean["rango_edad"] = pd.cut(
    df_clean["edad"],
    bins=[17, 29, 39, 49, 65],
    labels=["18-29", "30-39", "40-49", "50+"]
).astype(str)

# 2. Ratio de retención: proporción de su experiencia total trabajada en esta empresa
df_clean["ratio_antiguedad_exp"] = np.where(
    df_clean["experiencia_anios"] > 0,
    (df_clean["antiguedad_empresa"] / df_clean["experiencia_anios"]).clip(0, 1),
    0.0
).round(3)

# 3. Indicador binario: 1 si supera las 15 horas extra al mes
df_clean["horas_extra_altas"] = (df_clean["horas_extra_mes"] > 15).astype(int)

display(df_clean[["id_empleado", "edad", "rango_edad", "ratio_antiguedad_exp", "horas_extra_altas"]].head())

,id_empleado,edad,rango_edad,ratio_antiguedad_exp,horas_extra_altas
0,1,26,18-29,0.000,0
1,2,41,40-49,0.947,0
2,3,42,40-49,0.900,1
3,4,52,50+,1.000,1
4,5,57,50+,0.968,0


In [4]:
schema_sql = """
DROP TABLE IF EXISTS empleados;

CREATE TABLE empleados (
    id_empleado          INTEGER PRIMARY KEY,
    edad                 INTEGER NOT NULL CHECK (edad BETWEEN 18 AND 80),
    experiencia_anios    INTEGER NOT NULL CHECK (experiencia_anios >= 0),
    nivel_educativo      INTEGER NOT NULL CHECK (nivel_educativo BETWEEN 1 AND 4),
    departamento         TEXT NOT NULL,
    ciudad               TEXT NOT NULL,
    genero               TEXT NOT NULL CHECK (genero IN ('F', 'M')),
    horas_extra_mes      INTEGER NOT NULL CHECK (horas_extra_mes >= 0),
    satisfaccion_laboral INTEGER NOT NULL CHECK (satisfaccion_laboral BETWEEN 1 AND 5),
    antiguedad_empresa   INTEGER NOT NULL CHECK (antiguedad_empresa >= 0),
    salario_mensual      REAL NOT NULL CHECK (salario_mensual > 0),
    rango_edad           TEXT NOT NULL,
    ratio_antiguedad_exp REAL NOT NULL CHECK (ratio_antiguedad_exp BETWEEN 0 AND 1),
    horas_extra_altas    INTEGER NOT NULL CHECK (horas_extra_altas IN (0, 1))
);

CREATE INDEX idx_empleados_departamento ON empleados(departamento);
CREATE INDEX idx_empleados_ciudad ON empleados(ciudad);
"""

# Opcional: guarda automáticamente el archivo sql/schema.sql para el entregable
SCHEMA_PATH.parent.mkdir(parents=True, exist_ok=True)
SCHEMA_PATH.write_text(schema_sql.strip(), encoding="utf-8")

# Crea la base de datos e inserta los datos limpios
with sqlite3.connect(DB_PATH) as conn:
    conn.executescript(schema_sql)
    df_clean.to_sql("empleados", conn, if_exists="append", index=False)

print(f"Esquema guardado en: {SCHEMA_PATH}")
print(f"Base generada en: {DB_PATH}")

Esquema guardado en: ..\sql\schema.sql
Base generada en: ..\data\empleados.db


In [5]:
with sqlite3.connect(DB_PATH) as conn:
    # 1. Verificar conteo total e integridad básica
    resumen = pd.read_sql("""
        SELECT COUNT(*) AS total_empleados,
               COUNT(DISTINCT id_empleado) AS ids_unicos,
               ROUND(MIN(salario_mensual), 2) AS salario_min,
               ROUND(MAX(salario_mensual), 2) AS salario_max
        FROM empleados;
    """, conn)

    # 2. Agregados por departamento
    por_depto = pd.read_sql("""
        SELECT departamento,
               COUNT(*) AS n_empleados,
               ROUND(AVG(salario_mensual), 2) AS salario_promedio,
               ROUND(AVG(satisfaccion_laboral), 2) AS satisfaccion_promedio,
               ROUND(AVG(ratio_antiguedad_exp), 2) AS retencion_promedio
        FROM empleados
        GROUP BY departamento
        ORDER BY salario_promedio DESC;
    """, conn)

    # 3. Agregados por ciudad
    por_ciudad = pd.read_sql("""
        SELECT ciudad,
               COUNT(*) AS n_empleados,
               ROUND(AVG(salario_mensual), 2) AS salario_promedio,
               SUM(horas_extra_altas) AS empleados_horas_extra_altas
        FROM empleados
        GROUP BY ciudad
        ORDER BY n_empleados DESC;
    """, conn)

display(resumen)
display(por_depto)
display(por_ciudad)

,total_empleados,ids_unicos,salario_min,salario_max
0,600,600,17779.99,79107.94


,departamento,n_empleados,salario_promedio,satisfaccion_promedio,retencion_promedio
0,IT,138,45549.70,2.99,0.76
1,Finanzas,101,43923.37,3.43,0.76
2,RH,58,43445.42,2.76,0.78
3,Operaciones,115,41837.29,3.17,0.79
4,Marketing,82,41750.13,2.95,0.81
5,Ventas,106,39208.39,3.04,0.75


,ciudad,n_empleados,salario_promedio,empleados_horas_extra_altas
0,CDMX,212,43527.34,51
1,GDL,122,42545.26,39
2,MTY,121,43585.72,41
3,PUE,88,41125.49,24
4,QRO,57,40729.98,18
